## Homework 3: Symbolic Music Generation Using Markov Chains

**Before starting the homework:**

Please run `pip install miditok` to install the [MiDiTok](https://github.com/Natooz/MidiTok) package, which simplifies MIDI file processing by making note and beat extraction more straightforward.

You’re also welcome to experiment with other MIDI processing libraries such as [mido](https://github.com/mido/mido), [pretty_midi](https://github.com/craffel/pretty-midi) and [miditoolkit](https://github.com/YatingMusic/miditoolkit). However, with these libraries, you’ll need to handle MIDI quantization yourself, for example, converting note-on/note-off events into beat positions and durations.

In [ ]:
# run this command to install MiDiTok
# ! pip install miditok

  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/159.0 kB ? eta -:--:--
   ------------------------- -------------- 102.4/159.0 kB


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ! pip install MIDIUtil
# 

     ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.0/1.0 MB 640.0 kB/s eta 0:00:02
     --- ------------------------------------ 0.1/1.0 MB 907.3 kB/s eta 0:00:02
     ------- -------------------------------- 0.2/1.0 MB 1.6 MB/s eta 0:00:01
     ---------------- ----------------------- 0.4/1.0 MB 2.6 MB/s eta 0:00:01
     ---------------------------------------  1.0/1.0 MB 5.3 MB/s eta 0:00:01
     ---------------------------------------- 1.0/1.0 MB 4.9 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for MIDIUtil: filename=midiutil-1.2.1-py


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# import required packages
import random
from glob import glob
from collections import defaultdict, Counter

import numpy as np
from numpy.random import choice

from symusic import Score
from miditok import REMI, TokenizerConfig
from midiutil import MIDIFile

c:\Users\Seojin Park\Desktop\Coding\CSE-153-HW3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# You can change the random seed but try to keep your results deterministic!
# If I need to make changes to the autograder it'll require rerunning your code,
# so it should ideally generate the same results each time.
random.seed(42)

In [8]:
# from google.colab import drive
# drive.mount('/content/drive')

### Load music dataset
We will use a subset of the [PDMX dataset](https://zenodo.org/records/14984509).

Please find the file `PDMX_subset.zip` in the homework spec.

All pieces are monophonic music (i.e. one melody line) in 4/4 time signature.

In [14]:
# midi_files = glob('/Users/comos/OneDrive/바탕 화면/CSE 153 HW3/PDMX_subset/PDMX_subset/*.mid')
# midi_files = glob('PDMX_subset/*.mid')
midi_files = glob('/Users/Seojin Park/Desktop/Coding/CSE-153-HW3/PDMX_subset/PDMX_subset/*.mid')

len(midi_files)

1000

### Train a tokenizer with the REMI method in MidiTok

In [15]:
config = TokenizerConfig(num_velocities=1, use_chords=False, use_programs=False)
tokenizer = REMI(config)
tokenizer.train(vocab_size=1000, files_paths=midi_files)

### Use the trained tokenizer to get tokens for each midi file
In REMI representation, each note will be represented with four tokens: `Position, Pitch, Velocity, Duration`, e.g. `('Position_28', 'Pitch_74', 'Velocity_127', 'Duration_0.4.8')`; a `Bar_None` token indicates the beginning of a new bar.

In [16]:
# e.g.:
midi = Score(midi_files[0])
tokens = tokenizer(midi)[0].tokens
tokens[:10]

['Bar_None',
 'Position_0',
 'Pitch_66',
 'Velocity_127',
 'Duration_0.4.8',
 'Position_4',
 'Pitch_62',
 'Velocity_127',
 'Duration_0.4.8',
 'Position_8']

1. Write a function to extract note pitch events from a midi file; and another extract all note pitch events from the dataset and output a dictionary that maps note pitch events to the number of times they occur in the files. (e.g. {60: 120, 61: 58, …}).

`note_extraction()`
- **Input**: a midi file

- **Output**: a list of note pitch events (e.g. [60, 62, 61, ...])

`note_frequency()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to the number of times they occur, e.g {60: 120, 61: 58, …}

In [12]:
def note_extraction(midi_file):
    # Q1a: Your code goes here
    score = Score(midi_file)
    # print(score)
    # get tokens and its at [0]
    tokens = tokenizer(score)[0].tokens
    temp = []
    
    # get all tokens starting with pitch
    for i in range(len(tokens)):
        if tokens[i].startswith("Pitch_"):
            temp.append(int(tokens[i].split("_")[1]))
    return temp
    # pass
# note_extraction

In [13]:
def note_frequency(midi_files):
    # Q1b: Your code goes here
    # use hashmap to count
    h = {}
    for i in midi_files:
        for j in note_extraction(i):
            h[j] = h.get(j,0) + 1
    return h
    pass

2. Write a function to normalize the above dictionary to produce probability scores (e.g. {60: 0.13, 61: 0.065, …})

`note_unigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to probabilities, e.g. {60: 0.13, 61: 0.06, …}

In [14]:
def note_unigram_probability(midi_files):
    note_counts = note_frequency(midi_files)
    unigramProbabilities = {}

    # Q2: Your code goes here
    # ...
    # counts = note_frequency(midi_files)
    temp = sum(note_counts.values())
    for i, j in note_counts.items():
        unigramProbabilities[i] = j / temp
    return unigramProbabilities

3. Generate a table of pairwise probabilities containing p(next_note | previous_note) values for the dataset; write a function that randomly generates the next note based on the previous note based on this distribution.

`note_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramTransitions`: key: previous_note, value: a list of next_note, e.g. {60:[62, 64, ..], 62:[60, 64, ..], ...} (i.e., this is a list of every other note that occured after note 60, every note that occured after note 62, etc.)

  - `bigramTransitionProbabilities`: key:previous_note, value: a list of probabilities for next_note in the same order of `bigramTransitions`, e.g. {60:[0.3, 0.4, ..], 62:[0.2, 0.1, ..], ...} (i.e., you are converting the values above to probabilities)

`sample_next_note()`
- **Input**: a note

- **Output**: next note sampled from pairwise probabilities

In [15]:
def note_bigram_probability(midi_files):
    # default dicts are meh
    # possible next notes, prob for each next note
    bigramTransitions = defaultdict(list)
    bigramTransitionProbabilities = defaultdict(list)

    # Q3a: Your code goes here
    # ...
    h = {}
    
    # count biagrams
    for i in midi_files:
        temp = note_extraction(i)
        for j in range(1, len(temp)):
            prev = temp[j-1]
            nxt = temp[j]
            if prev not in h:
                h[prev] = {}
            if nxt not in h[prev]:
                h[prev][nxt] = 0
            h[prev][nxt] += 1
            
            
            
            
    # make transistions into biagrams, using i,j like before for key, value
    for i, j in h.items():
        res = sum(j.values())
        bigramTransitions[i] = list(j.keys())
        temp = []
        
        for count in j.values():
            temp.append(count / res)
        bigramTransitionProbabilities[i] = temp

    return bigramTransitions, bigramTransitionProbabilities

In [16]:
bigramTransitions = {}
bigramTransitionProbabilities = {}
bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
def sample_next_note(note):
    # Q3b: Your code goes here
    # next note sampled from pairwise probabilities is output
    # list of possible next notes using glpobal variable
    nxt = bigramTransitions[note]
    # probabailiys
    prob = bigramTransitionProbabilities[note]
    # get random and return
    return random.choices(nxt, weights=prob, k=1)[0]
    # return res
    pass

4. Write a function to calculate the perplexity of your model on a midi file.

    The perplexity of a model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-1})))$

    where $p(w_1|w_0) = p(w_1)$, $p(w_i|w_{i-1}) (i>1)$ refers to the pairwise probability p(next_note | previous_note).

`note_bigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [ ]:
def note_bigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)

    # Q4: Your code goes here
    # Can use regular numpy.log (i.e., natural logarithm)
    # look at module 3 page 74 for future ref
    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return float("inf")
    res = 0
    for i in range(len(notes)):
        # for w1
        if i == 0:
            # im doing 0.0001 bc we cant do log by 0 
            temp = unigramProbabilities.get(notes[i], 1e-9)
        else:
            prev_note = notes[i - 1]
            curr_note = notes[i]
            if prev_note in bigramTransitions and curr_note in bigramTransitions[prev_note]:
                idx = bigramTransitions[prev_note].index(curr_note)
                temp = bigramTransitionProbabilities[prev_note][idx]
            else:
                temp = 1e-9
        res += np.log(temp)
    res = np.exp(-res / len(notes))
    return res


5. Implement a second-order Markov chain, i.e., one which estimates p(next_note | next_previous_note, previous_note); write a function to compute the perplexity of this new model on a midi file.

    The perplexity of this model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-2}, w_{i-1})))$

    where $p(w_1|w_{-1}, w_0) = p(w_1)$, $p(w_2|w_0, w_1) = p(w_2|w_1)$, $p(w_i|w_{i-2}, w_{i-1}) (i>2)$ refers to the probability p(next_note | next_previous_note, previous_note).


`note_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramTransitions`: key - (next_previous_note, previous_note), value - a list of next_note, e.g. {(60, 62):[64, 66, ..], (60, 64):[60, 64, ..], ...}

  - `trigramTransitionProbabilities`: key: (next_previous_note, previous_note), value: a list of probabilities for next_note in the same order of `trigramTransitions`, e.g. {(60, 62):[0.2, 0.2, ..], (60, 64):[0.4, 0.1, ..], ...}

`note_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [ ]:
def note_trigram_probability(midi_files):
    trigramTransitions = defaultdict(list)
    trigramTransitionProbabilities = defaultdict(list)

    # Q5a: Your code goes here
    # ...
    # just use counter ig even if I dont use it as much
    h = defaultdict(Counter)

    #I love hashmaps
    for midi_file in midi_files:
        notes = note_extraction(midi_file)

        for i in range(2, len(notes)):
            context = (notes[i - 2], notes[i - 1])
            next_note = notes[i]
            h[context][next_note] += 1
            
    # I think it still works without items but its whatever
    for i,j in h.items():
        temp = sum(j.values())
        for a,b in j.items():
            trigramTransitions[i].append(a) 
            trigramTransitionProbabilities[i].append(b/ temp)
            
    return trigramTransitions, trigramTransitionProbabilities

In [ ]:
def note_trigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    # Q5b: Your code goes here
    
    
    
    
    notes = note_extraction(midi_file)
    # just making sure again
    if len(notes) == 0:
        return float("inf")

    res = 0
    for i in range(len(notes)):
        curr_note = notes[i]
        # check if we have 0, 1 or all remaining notes with if
        if i ==0:
            temp = unigramProbabilities.get(curr_note, 1e-9)

        elif i == 1:
            prev_note = notes[i - 1]
            if prev_note in bigramTransitions and curr_note in bigramTransitions[prev_note]:
                idx = bigramTransitions[prev_note].index(curr_note)
                temp = bigramTransitionProbabilities[prev_note][idx]
            else:
                temp = 1e-9


        # everything else P(currrent | prev2, prev1)
        else:
            context = (notes[i - 2], notes[i - 1])
            if context in trigramTransitions and curr_note in trigramTransitions[context]:
                index = trigramTransitions[context].index(curr_note)
                temp = trigramTransitionProbabilities[context][index]
            else:
                # basically does 0.,000001
                temp = 1e-9
        res += np.log(temp)



    res = np.exp(-res / len(notes))
    return res


6. Our model currently doesn’t have any knowledge of beats. Write a function that extracts beat lengths and outputs a list of [(beat position; beat length)] values.

    Recall that each note will be encoded as `Position, Pitch, Velocity, Duration` using REMI. Please keep the `Position` value for beat position, and convert `Duration` to beat length using provided lookup table `duration2length` (see below).

    For example, for a note represented by four tokens `('Position_24', 'Pitch_72', 'Velocity_127', 'Duration_0.4.8')`, the extracted (beat position; beat length) value is `(24, 4)`.

    As a result, we will obtain a list like [(0,8),(8,16),(24,4),(28,4),(0,4)...], where the next beat position is the previous beat position + the beat length. As we divide each bar into 32 positions by default, when reaching the end of a bar (i.e. 28 + 4 = 32 in the case of (28, 4)), the beat position reset to 0.

In [ ]:
duration2length = {
    '0.2.8': 2,  # sixteenth note, 0.25 beat in 4/4 time signature
    '0.4.8': 4,  # eighth note, 0.5 beat in 4/4 time signature
    '1.0.8': 8,  # quarter note, 1 beat in 4/4 time signature
    '2.0.8': 16, # half note, 2 beats in 4/4 time signature
    '4.0.4': 32, # whole note, 4 beats in 4/4 time signature
}

`beat_extraction()`
- **Input**: a midi file

- **Output**: a list of (beat position; beat length) values

In [ ]:
def beat_extraction(midi_file):
    # Q6: Your code goes here
    # notes = note_extraction(midi_file)
    # im so stupid we cant reuse note extrreaction
    res = []
    i = 0
    score = Score(midi_file)
    tokens = tokenizer(score)[0].tokens
    while i < len(tokens):
        if tokens[i].startswith("Position_"):
            pos_token = tokens[i]
            dur_token = tokens[i + 3]

            pos = int(pos_token.split('_')[1])
            length = duration2length.get(dur_token.split("Duration_")[1], 0)

            res.append((pos, length))
            i += 4
        else:
            i += 1

    return res
    pass

7. Implement a Markov chain that computes p(beat_length | previous_beat_length) based on the above function.

`beat_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatTransitions`: key: previous_beat_length, value: a list of beat_length, e.g. {4:[8, 2, ..], 8:[8, 4, ..], ...}

  - `bigramBeatTransitionProbabilities`: key - previous_beat_length, value - a list of probabilities for beat_length in the same order of `bigramBeatTransitions`, e.g. {4:[0.3, 0.2, ..], 8:[0.4, 0.4, ..], ...}

In [1]:
def beat_bigram_probability(midi_files):
    bigramBeatTransitions = defaultdict(list)
    bigramBeatTransitionProbabilities = defaultdict(list)

    # Q7: Your code goes here
    # ... We do p(lent | len t - 1)
    # use dd and lambda
    h = defaultdict(lambda: defaultdict(int))
    for i in midi_files:
        # get beats, tuples of pos, len
        beats = beat_extraction(i)
        # count bigram trans
        for j in range(1, len(beats)):
            prev_len = beats[j-1][1]
            curr_len = beats[j][1]
            h[prev_len][curr_len] += 1
            
    # make into probs   
    for i, j in h.items():
        total = sum(j.values())
        bigramBeatTransitions[i] = list(j.keys())
        bigramBeatTransitionProbabilities[i] = [
            c / total for c in j.values()
        ]
    return bigramBeatTransitions, bigramBeatTransitionProbabilities

8. Implement a function to compute p(beat length | beat position), and compute the perplexity of your models from Q7 and Q8. For both models, we only consider the probabilities of predicting the sequence of **beat lengths**.

`beat_pos_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatPosTransitions`: key - beat_position, value - a list of beat_length

  - `bigramBeatPosTransitionProbabilities`: key - beat_position, value - a list of probabilities for beat_length in the same order of `bigramBeatPosTransitions`

`beat_bigram_perplexity()`
- **Input**: a midi file

- **Output**: two perplexity values correspond to the models in Q7 and Q8, respectively

In [ ]:
def beat_pos_bigram_probability(midi_files):
    bigramBeatPosTransitions = defaultdict(list)
    bigramBeatPosTransitionProbabilities = defaultdict(list)

    # Q8a: Your code goes here
    # ...
    h = defaultdict(lambda: defaultdict(int))
    # count
    for i in midi_files:
        beats = beat_extraction(i)
        for j in beats:
            pos = j[0]
            length = j[1]
            h[pos][length] += 1

    # convert again
    for i, j in h.items():
        total = sum(j.values())
        bigramBeatPosTransitions[i] = list(j.keys())
        bigramBeatPosTransitionProbabilities[i] = [
            c / total for c in j.values()
        ]

    return bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities

In [ ]:
def beat_bigram_perplexity(midi_file):
    bigramBeatTransitions, bigramBeatTransitionProbabilities = beat_bigram_probability(midi_files)
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    # Q8b: Your code goes here
    # Hint: one more probability function needs to be computed

    # perplexity for Q7
    perplexity_Q7 = None

    # perplexity for Q8
    perplexity_Q8 = None


    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return float("inf"), float("inf")
    
    # helper function to get probabailtuiy
    def helper(mapping, probs, key1, key2, default=1e-9):
        if key1 in mapping and key2 in mapping[key1]:
            idx = mapping[key1].index(key2)
            return probs[key1][idx]
        return default
    # q7 perp
    res = 0
    for i in range(len(beats)):
        curr_len = beats[i][1]

        if i == 0:
            prob = 1 / len(duration2length)
        else:
            prev_len = beats[i -1][1]
            prob = helper(bigramBeatTransitions,bigramBeatTransitionProbabilities, prev_len, curr_len)

        res += np.log(prob)

    perplexity_Q7 = np.exp(-res / len(beats))
    
    # q8 perp
    res2 = 0
    for pos, curr_len in beats:
        prob = helper(bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities, pos, curr_len)
        res2 += np.log(prob)

    perplexity_Q8 = np.exp(-res2 / len(beats))
    
    return perplexity_Q7, perplexity_Q8

9. Implement a Markov chain that computes p(beat_length | previous_beat_length, beat_position), and report its perplexity.

`beat_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramBeatTransitions`: key: (previous_beat_length, beat_position), value: a list of beat_length

  - `trigramBeatTransitionProbabilities`: key: (previous_beat_length, beat_position), value: a list of probabilities for beat_length in the same order of `trigramBeatTransitions`

`beat_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [ ]:
def beat_trigram_probability(midi_files):
    trigramBeatTransitions = defaultdict(list)
    trigramBeatTransitionProbabilities = defaultdict(list)

    # Q9a: Your code goes here
    # ... hashmap: h[(prev_len, pos)][curr_len]
    h = defaultdict(lambda: defaultdict(int))

    # count
    for midi_file in midi_files:
        beats = beat_extraction(midi_file)
        for i in range(1, len(beats)):
            prev_len = beats[i - 1][1]
            pos = beats[i][0]
            curr_len = beats[i][1]

            context = (prev_len, pos)
            h[context][curr_len] += 1

    # convert again agian
    for i, j in h.items():
        total = sum(j.values())
        trigramBeatTransitions[i] = list(j.keys())
        trigramBeatTransitionProbabilities[i] = [c / total for c in j.values()]
    return trigramBeatTransitions, trigramBeatTransitionProbabilities

In [ ]:
def beat_trigram_perplexity(midi_file):
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    trigramBeatTransitions, trigramBeatTransitionProbabilities = beat_trigram_probability(midi_files)
    # Q9b: Your code goes here
    
    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return float("inf")
    res = 0

    for i in range(len(beats)):
        temp = beats[i][1]

        # first beat
        if i == 0:
            prob = 1 / len(duration2length)
        # trigram probability
        elif i == 1:
            pos = beats[i][0]
            if pos in bigramBeatPosTransitions and temp in bigramBeatPosTransitions[pos]:
                idx = bigramBeatPosTransitions[pos].index(temp)
                prob = bigramBeatPosTransitionProbabilities[pos][idx]
            else:
                # just use smoothing ig
                prob = 1 / len(duration2length)
        else:
            prev_len = beats[i-1][1]
            pos = beats[i][0]
            context = (prev_len, pos)
            if context in trigramBeatTransitions and temp in trigramBeatTransitions[context]:
                idx = trigramBeatTransitions[context].index(temp)
                prob = trigramBeatTransitionProbabilities[context][idx]
                # fall back to 0.000001
            else:
                prob = 1 / len(duration2length)
                
                
                
        res += np.log(prob)
    return np.exp(-res / len(beats))

10. Use the model from Q5 to generate N notes, and the model from Q8 to generate beat lengths for each note. Save the generated music as a midi file (see code from workbook1) as q10.mid. Remember to reset the beat position to 0 when reaching the end of a bar.

`music_generate`
- **Input**: target length, e.g. 500

- **Output**: a midi file q10.mid

Note: the duration of one beat in MIDIUtil is 1, while in MidiTok is 8. Divide beat length by 8 if you use methods in MIDIUtil to save midi files.

In [ ]:
def music_generate(length):
    # sample notes
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    # Q10: Your code goes here ...
    sampled_notes = []

    # sample beats
    sampled_beats = []

    # save the generated music as a midi file


    beatPosTransitions, beatPosProbs = beat_pos_bigram_probability(midi_files)
    notesList = list(unigramProbabilities.keys())
    probsList = list(unigramProbabilities.values())
    firstNote = random.choices(notesList, weights=probsList)[0]

    prev_prev_note = firstNote
    prev_note = firstNote

    sampled_notes.append(firstNote)

    # init beat
    temp = 0

    for i in range(length):
        # beat
        if temp in beatPosTransitions:
            lengths = beatPosTransitions[temp]
            probs = beatPosProbs[temp]
            beatLen = random.choices(lengths, weights=probs)[0]
        else:
            beatLen = 8

        sampled_beats.append((temp, beatLen))

        #update position
        temp = (temp + beatLen) % 32

        # note
        if i == 0:
            nextNote = firstNote
        elif i == 1:
            nextNote = sample_next_note(prev_note)
        else:
            context = (prev_prev_note, prev_note)
            if context in trigramTransitions:
                nxts = trigramTransitions[context]
                probs = trigramTransitionProbabilities[context]
                nextNote = random.choices(nxts, weights=probs)[0]
            else:
                nextNote = sample_next_note(prev_note)

        sampled_notes.append(nextNote)
        prev_prev_note, prev_note = prev_note, nextNote


    # write to file
    midi = MIDIFile(1)
    track = 0
    time = 0
    channel = 0
    volume = 100

    for note, (pos, length) in zip(sampled_notes, sampled_beats):
        duration = length / 8
        midi.addNote(track, channel, note, time, duration, volume)
        time += duration

    with open("q10.mid", "wb") as f:
        midi.writeFile(f)